In [ ]:
import json
from pathlib import Path
import os
import numpy as np
import torch
from matplotlib import pyplot as plt

from robomimic.utils import file_utils as FileUtils
from robomimic.utils import obs_utils as ObsUtils

from world_model_utils import (
    build_state_from_obs_dict,
    load_init_state_json,
    load_model_for_eval,
    plot_rzz_world_model_branches,
    rollout_policy_for_rzz_analysis,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

### Finding the right env seed to test the diffusion guidance
could use env_seeds.sub to roll out from more starting positions and find the ones where policy is diverse and model is accurate

In [ ]:
# Prepare policy, environment, and learned world model
CKPT_SOURCE = "gilbreth"  # "local" or "gilbreth"
CKPT_PATH = (
    "/scratch/gilbreth/zoellner/diffusion_runs/can_mh_lowdim/20260209180642/models/model_epoch_2000.pth"
    if CKPT_SOURCE == "gilbreth"
    else "../models/can/model_epoch_2000.pth"
)

ckpt = torch.load(CKPT_PATH, map_location="cpu")
cfg = json.loads(ckpt["config"])
algo_override = json.loads(Path("../configs/diffusion_policy.json").read_text())["algo"]
cfg["algo"]["ddpm"] = algo_override["ddpm"]
cfg["algo"]["ddim"] = algo_override["ddim"]
ckpt["config"] = json.dumps(cfg)
ObsUtils.initialize_obs_utils_with_obs_specs(cfg["observation"]["modalities"])

env, _ = FileUtils.env_from_checkpoint(
    ckpt_dict=ckpt,
    render=True,
    render_offscreen=True,
)
policy, _ = FileUtils.policy_from_checkpoint(
    ckpt_dict=ckpt,
    device=device,
    verbose=True,
)

predictor_learned, stats, _, eval_meta = load_model_for_eval(
    model_or_run_path="../models/dynamics/baseline_mlp",
    predictor_kind="learned",
    device=device,
    load_val_trajectories=False,
)

print(f"Policy ckpt: {CKPT_PATH}")
print(f"World-model run dir: {eval_meta['run_dir']}")

In [ ]:
potential_seed_run_dirs = [
    Path("../outputs/can_rollouts/snr/n16h300r60g1bYfYv1"),
    Path("../outputs/can_rollouts/snr/n16h300r60g1bYfYv4"),
]

In [ ]:
print(device)

In [ ]:
# Rollout each potential seed init-state 3 times and overlay world-model branch predictions per seed
import json
import pickle
from datetime import datetime

horizon = 300
branch_horizon = 8
branch_stride = 8
video_skip = 1
camera_name = "frontview"
n_repeats_per_seed = 3

# Create timestamped output directory for this run
base_analysis_dir = Path("../outputs/can_rollouts/diffusion_guidance_checks")
base_analysis_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
analysis_dir = base_analysis_dir / timestamp
analysis_dir.mkdir(parents=True, exist_ok=True)
videos_dir = analysis_dir / "videos"
videos_dir.mkdir(parents=True, exist_ok=True)
plots_dir = analysis_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving results to: {analysis_dir.resolve()}")

seed_rollouts = {}
for run_dir in potential_seed_run_dirs:
    init_state_path = run_dir / "init_state.json"
    if not init_state_path.exists():
        print(f"Skipping {run_dir}: no init_state.json")
        continue

    init_state = load_init_state_json(str(init_state_path))
    seed_rollouts[run_dir.name] = []

    fig, ax = plt.subplots(figsize=(12, 5))
    for rep in range(n_repeats_per_seed):
        video_path = videos_dir / f"{run_dir.name}_rep{rep+1}_h{horizon}_{camera_name}.mp4"

        rollout_result = rollout_policy_for_rzz_analysis(
            env=env,
            policy=policy,
            build_state_from_obs_fn=build_state_from_obs_dict,
            horizon=horizon,
            video_path=str(video_path),
            video_skip=video_skip,
            camera_name=camera_name,
            mujoco_body_name="Can_main",
            init_state=init_state,
        )

        seed_rollouts[run_dir.name].append(rollout_result)
        print(
            f"{run_dir.name} rep{rep+1}: steps={rollout_result['steps_executed']} ",
            f"return={rollout_result['total_reward']:.4f} success={rollout_result['success']}"
        )

        plot_rzz_world_model_branches(
            states=rollout_result["states"],
            actions=rollout_result["actions"],
            predictor=predictor_learned,
            stats=stats,
            device=device,
            branch_horizon=branch_horizon,
            branch_stride=branch_stride,
            title=f"{run_dir.name}: 3 repeated rollouts | true Rzz + world-model branches",
            ax=ax,
            show=False,
            true_label=f"true Rzz (rep {rep+1})",
            branch_label=f"branches (rep {rep+1})",
            branch_alpha=0.7,
            show_legend=True,
        )

    plt.show()
    
    # Save the plot for this seed
    plot_path = plots_dir / f"{run_dir.name}_rzz_world_model_branches.png"
    fig.savefig(plot_path, dpi=120, bbox_inches="tight")
    print(f"✓ Saved plot to {plot_path}")
    plt.close(fig)

print(f"Finished {len(seed_rollouts)} seed groups, {n_repeats_per_seed} rollouts each.")

# ---------- Save rollout data and metadata ----------
# Save the complete rollout results dict for later analysis
rollout_data_path = analysis_dir / "seed_rollouts_full.pkl"
with open(rollout_data_path, "wb") as f:
    pickle.dump(seed_rollouts, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"✓ Saved rollout data to {rollout_data_path}")

# Save experiment metadata
metadata = {
    "timestamp": timestamp,
    "n_repeats_per_seed": n_repeats_per_seed,
    "horizon": horizon,
    "branch_horizon": branch_horizon,
    "branch_stride": branch_stride,
    "seed_groups": list(seed_rollouts.keys()),
    "num_seed_groups": len(seed_rollouts),
    "camera_name": camera_name,
}
metadata_path = analysis_dir / "experiment_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2))
print(f"✓ Saved metadata to {metadata_path}")

print(f"\n📊 All results saved to: {analysis_dir.resolve()}")

### Rollout with Diffusion guidance ?

In [ ]:
# Seed all RNG for reproducibility
import random

def reseed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# Sweep synthetic guidance scale from 0 to 100

def first_up_others_down_guidance(states, actions, other_weight=0.0):
    objective = actions[:, 0, 0].mean() - other_weight * (actions[:, 0, 1:] ** 2).mean()
    return torch.autograd.grad(objective, actions, retain_graph=False, create_graph=False)[0]

scales = [0, 0.1, 0.5, 1, 2, 5, 10, 20, 50, 100]
results = []

policy.start_episode()
obs = env.reset_to(init_state)

reseed(123)
policy.start_episode()
a_ref = policy(ob=obs)
a0_ref = float(a_ref[0])
other_abs_ref = float(np.mean(np.abs(a_ref[1:])))

for s in scales:
    reseed(123)
    policy.start_episode()
    a = policy(
        ob=obs,
        guidance_function=first_up_others_down_guidance,
        guidance_type="diffusion",
        guidance_scale=float(s),
    )
    a0 = float(a[0])
    mean_other_delta = float(np.mean(a[1:] - a_ref[1:]))
    other_abs = float(np.mean(np.abs(a[1:])))
    results.append((float(s), a0, a0 - a0_ref, mean_other_delta, other_abs - other_abs_ref))

print("baseline a[0]:", a0_ref)
print("baseline mean abs(other dims):", other_abs_ref)
print("scale | guided a[0] | delta a[0] | mean delta other | delta mean abs(other)")
for s, a0, d0, d_other, d_other_abs in results:
    print(f"{s:>5.1f} | {a0:+.6f} | {d0:+.6f} | {d_other:+.6f} | {d_other_abs:+.6f}")

In [ ]:
# Sanity test: one DP call without guidance vs one DP call with world-model guidance (no rollout)

def _rotation_6d_to_rzz_torch(rot6d: torch.Tensor) -> torch.Tensor:
    r1 = rot6d[..., 0:3]
    r2 = rot6d[..., 3:6]

    r1 = r1 / (torch.linalg.norm(r1, dim=-1, keepdim=True) + 1e-8)
    proj = torch.sum(r2 * r1, dim=-1, keepdim=True)
    r2 = r2 - proj * r1
    r2 = r2 / (torch.linalg.norm(r2, dim=-1, keepdim=True) + 1e-8)
    r3 = torch.cross(r1, r2, dim=-1)

    # Rzz is row=2,col=2 => z-component of 3rd column
    return r3[..., 2]


def make_world_model_guidance_fn(predictor, stats, device, rollout_steps=8):

    state_mean = torch.tensor(stats["state_mean"], device=device, dtype=torch.float32).unsqueeze(0)
    state_std = torch.tensor(stats["state_std"], device=device, dtype=torch.float32).unsqueeze(0)
    action_mean = torch.tensor(stats["action_mean"], device=device, dtype=torch.float32).unsqueeze(0)
    action_std = torch.tensor(stats["action_std"], device=device, dtype=torch.float32).unsqueeze(0)
    delta_mean = torch.tensor(stats["delta_mean"], device=device, dtype=torch.float32).unsqueeze(0)
    delta_std = torch.tensor(stats["delta_std"], device=device, dtype=torch.float32).unsqueeze(0)

    def guidance(states, actions):
        # states is a dict with keys like 'object', 'robot0_joint_pos', etc.
        # Each value has shape [B, T, ...] where T=2 (last two timesteps)
        # actions: [B, 16, 7]
        
        # Extract obs_dict from states - take first batch item, last timestep
        obs_dict = {
            key: (val.detach().cpu().numpy() if torch.is_tensor(val) else np.asarray(val))[0, -1]
            for key, val in states.items()
        }
        
        # Build 29D state from observation dict (uses existing world_model_utils helper)
        state_now_np = build_state_from_obs_dict(obs_dict)
        s = torch.tensor(state_now_np, device=device, dtype=torch.float32).unsqueeze(0)

        # Rollout first h steps through world model and compute gradient of mean Rzz
        bsz = actions.shape[0]
        h = min(int(rollout_steps), actions.shape[1])
        rzz_traj = []
        
        for t in range(h):
            a_t = actions[:, t, :]
            s_n = (s - state_mean) / state_std
            a_n = (a_t - action_mean) / action_std
            d_n = predictor(s_n, a_n)
            d = d_n * delta_std + delta_mean
            s = s + d

            # Extract and track predicted can Rzz
            q_can_6d = s[:, 12:18]
            rzz_t = _rotation_6d_to_rzz_torch(q_can_6d)
            rzz_traj.append(rzz_t)

        # Maximize mean predicted Rzz over the first h rollout steps
        rzz_stack = torch.stack(rzz_traj, dim=1)  # [B, h]
        objective = rzz_stack.mean()

        # Backprop to get gradient w.r.t. actions
        grad = torch.autograd.grad(objective, actions, retain_graph=False, create_graph=False)[0]
        return grad

    return guidance


# Setup test - use init state from first seed
init_state_path = potential_seed_run_dirs[0] / "init_state.json"
init_state = load_init_state_json(str(init_state_path))

policy.start_episode()
obs = env.reset_to(init_state)

guidance_fn = make_world_model_guidance_fn(
    predictor=predictor_learned,
    stats=stats,
    device=device,
    rollout_steps=8,
)

reseed(42)
# Baseline sample
policy.start_episode()
act_no_guidance = policy(ob=obs)

# Guided sample (guidance_scale applied in diffusion denoising loop)
reseed(42)
policy.start_episode()
guidance_scale = 5
act_with_guidance = policy(
    ob=obs,
    guidance_function=guidance_fn,
    guidance_type="diffusion",
    guidance_scale=guidance_scale,
 )

print("No-guidance action:", act_no_guidance)
print("With-guidance action:", act_with_guidance)
print("L2 diff:", float(np.linalg.norm(act_with_guidance - act_no_guidance)))
print("max|diff|:", float(np.max(np.abs(act_with_guidance - act_no_guidance))))

In [ ]:
# Extract denoising tracking buffers and compute norm summary statistics
import numpy as np
import matplotlib.pyplot as plt

inner_policy = policy.policy

# Get tracking buffers from the inner policy
diffusion_list = inner_policy.diffusion_list
guidance_list = inner_policy.guidance_list
corrections_list = inner_policy.corrections_list

print(f"Denoising tracking buffer lengths:")
print(f"  diffusion_list: {len(diffusion_list)}, shape per entry: {diffusion_list[0].shape}")
print(f"  guidance_list: {len(guidance_list)}, shape per entry: {guidance_list[0].shape}")
print(f"  corrections_list: {len(corrections_list)}, shape per entry: {corrections_list[0].shape}")

# Convert to numpy for easier manipulation
diffusion_norms = []
guidance_norms = []
ratio_norms = []
guidance_norms_8 = []  # First 8 actions only
diffusion_norms_8 = []

for i in range(len(diffusion_list)):
    diff = diffusion_list[i].detach().cpu().numpy()  # Shape: [1, 16, 7]
    guid = guidance_list[i].detach().cpu().numpy()   # Shape: [1, 16, 7]
    
    # Full tensor norms
    diff_norm = np.linalg.norm(diff)
    guid_norm = np.linalg.norm(guid)
    
    diffusion_norms.append(diff_norm)
    guidance_norms.append(guid_norm)
    ratio_norms.append(guid_norm / (diff_norm + 1e-8))
    
    # First 8 actions only
    diff_norm_8 = np.linalg.norm(diff[:, :8, :])
    guid_norm_8 = np.linalg.norm(guid[:, :8, :])
    
    diffusion_norms_8.append(diff_norm_8)
    guidance_norms_8.append(guid_norm_8)

diffusion_norms = np.array(diffusion_norms)
guidance_norms = np.array(guidance_norms)
ratio_norms = np.array(ratio_norms)
diffusion_norms_8 = np.array(diffusion_norms_8)
guidance_norms_8 = np.array(guidance_norms_8)

print(f"\n=== NORM SUMMARY ===")
print(f"Full 16x7 action tensors:")
print(f"  Diffusion norms: min={diffusion_norms.min():.4f}, mean={diffusion_norms.mean():.4f}, max={diffusion_norms.max():.4f}")
print(f"  Guidance norms:  min={guidance_norms.min():.4f}, mean={guidance_norms.mean():.4f}, max={guidance_norms.max():.4f}")
print(f"  Ratio (guidance/diffusion): min={ratio_norms.min():.4f}, mean={ratio_norms.mean():.4f}, max={ratio_norms.max():.4f}")
print(f"\nFirst 8x7 action tensors only:")
print(f"  Diffusion norms: min={diffusion_norms_8.min():.4f}, mean={diffusion_norms_8.mean():.4f}, max={diffusion_norms_8.max():.4f}")
print(f"  Guidance norms:  min={guidance_norms_8.min():.4f}, mean={guidance_norms_8.mean():.4f}, max={guidance_norms_8.max():.4f}")

print(f"\n=== INTERPRETATION ===")
mean_ratio = ratio_norms.mean()
if mean_ratio < 0.01:
    print(f"⚠️  Guidance is VERY WEAK (mean ratio {mean_ratio:.4f} << 0.01)")
elif 0.01 <= mean_ratio <= 0.3:
    print(f"✓ Guidance is in REASONABLE REGIME (mean ratio {mean_ratio:.4f} in [0.01, 0.3])")
elif mean_ratio > 1.0:
    print(f"⚠️  Guidance is DOMINATING (mean ratio {mean_ratio:.4f} > 1.0) - may override prior too much")
else:
    print(f"✓ Guidance is in MODERATE REGIME (mean ratio {mean_ratio:.4f})")

num_dominant = np.sum(ratio_norms >= 1.0)
print(f"Number of steps where ratio >= 1.0: {num_dominant}/{len(ratio_norms)}")


In [ ]:
# Minimal recommended plots: norms and ratio vs denoising step
fig, axes = plt.subplots(3, 1, figsize=(10, 8))

steps = np.arange(len(diffusion_norms))

# Plot 1: Diffusion norm
axes[0].plot(steps, diffusion_norms, 'b-o', label='Full 16x7', linewidth=2, markersize=6)
axes[0].plot(steps, diffusion_norms_8, 'b--s', label='First 8x7', linewidth=1.5, markersize=4, alpha=0.6)
axes[0].set_ylabel('||diffusion_i||', fontsize=12)
axes[0].set_title('Diffusion Norm vs Denoising Step', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot 2: Guidance norm
axes[1].plot(steps, guidance_norms, 'r-o', label='Full 16x7', linewidth=2, markersize=6)
axes[1].plot(steps, guidance_norms_8, 'r--s', label='First 8x7', linewidth=1.5, markersize=4, alpha=0.6)
axes[1].set_ylabel('||guidance_i||', fontsize=12)
axes[1].set_title('Guidance Norm vs Denoising Step', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Plot 3: Ratio
axes[2].plot(steps, ratio_norms, 'g-o', label='||guidance_i|| / ||diffusion_i||', linewidth=2, markersize=6)
axes[2].axhline(y=0.01, color='orange', linestyle='--', linewidth=1.5, label='Weak threshold (0.01)')
axes[2].axhline(y=0.3, color='purple', linestyle='--', linewidth=1.5, label='Upper reasonable (0.3)')
axes[2].axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='Dangerous threshold (1.0)')
axes[2].set_xlabel('Denoising Step Index', fontsize=12)
axes[2].set_ylabel('Guidance / Diffusion Ratio', fontsize=12)
axes[2].set_title('Guidance Strength Ratio vs Denoising Step', fontsize=13, fontweight='bold')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3, which='both')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/guidance_artifacts/norm_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved plot to ../outputs/guidance_artifacts/norm_analysis.png")


In [ ]:
# Per-dimension and per-timestep analysis
print("=" * 70)
print("PER-DIMENSION ANALYSIS (mean |value| across all time steps and all denoising steps)")
print("=" * 70)

per_dim_diffusion = []
per_dim_guidance = []

for d in range(7):
    diff_vals = []
    guid_vals = []
    for i in range(len(diffusion_list)):
        diff = np.abs(diffusion_list[i].detach().cpu().numpy()[0, :, d])
        guid = np.abs(guidance_list[i].detach().cpu().numpy()[0, :, d])
        diff_vals.extend(diff.tolist())
        guid_vals.extend(guid.tolist())
    
    mean_diff = np.mean(diff_vals)
    mean_guid = np.mean(guid_vals)
    per_dim_diffusion.append(mean_diff)
    per_dim_guidance.append(mean_guid)
    
    print(f"Dim {d}: diffusion={mean_diff:.6f}, guidance={mean_guid:.6f}, ratio={mean_guid/(mean_diff+1e-8):.4f}")

print("\n" + "=" * 70)
print("PER-TIMESTEP ANALYSIS (mean |value| across action dims, averaged over denoising steps)")
print("=" * 70)

for t in range(16):
    diff_vals = []
    guid_vals = []
    for i in range(len(diffusion_list)):
        diff = np.abs(diffusion_list[i].detach().cpu().numpy()[0, t, :])
        guid = np.abs(guidance_list[i].detach().cpu().numpy()[0, t, :])
        diff_vals.extend(diff.tolist())
        guid_vals.extend(guid.tolist())
    
    mean_diff = np.mean(diff_vals)
    mean_guid = np.mean(guid_vals)
    marker = " <- TARGET" if t < 8 else ""
    print(f"Time {t:2d}: diffusion={mean_diff:.6f}, guidance={mean_guid:.6f}, ratio={mean_guid/(mean_diff+1e-8):.4f}{marker}")

print("\n" + "=" * 70)
print("SUMMARY INTERPRETATION")
print("=" * 70)
print(f"Overall mean ratio across all dims/timesteps: {np.mean(per_dim_guidance)/(np.mean(per_dim_diffusion)+1e-8):.4f}")
print(f"This is BELOW 0.01 threshold -> guidance effect may be too subtle")
print(f"\nRecommendation: Consider increasing guidance_scale or checking objective function gradient")


In [ ]:
# Seeded rollout sweep with full denoising tracking capture and artifact export
import copy
import json
import pickle
from pathlib import Path

import imageio.v2 as imageio
import numpy as np

from world_model_utils import get_mujoco_rzz_and_pos


def _to_numpy_tree(x):
    if isinstance(x, dict):
        return {k: _to_numpy_tree(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_to_numpy_tree(v) for v in x]
    if torch.is_tensor(x):
        return x.detach().cpu().numpy().copy()
    if isinstance(x, np.ndarray):
        return x.copy()
    return x


def _clone_tracking_list(tracking_list, inner_policy):
    # Each entry is typically [1, 16, 7], stacked across denoising steps
    if len(tracking_list) > 0:
        return np.stack([t.detach().cpu().numpy().copy() for t in tracking_list], axis=0)

    # Unguided path may not populate lists; synthesize zeros for consistent logging
    denoise_steps = 0
    if hasattr(inner_policy, "noise_scheduler_ddim") and hasattr(inner_policy.noise_scheduler_ddim, "timesteps"):
        denoise_steps = int(len(inner_policy.noise_scheduler_ddim.timesteps))
    if denoise_steps <= 0:
        denoise_steps = 10

    pred_horizon = int(getattr(inner_policy, "pred_horizon", 16))
    ac_dim = int(getattr(inner_policy, "ac_dim", 7))
    return np.zeros((denoise_steps + 1, 1, pred_horizon, ac_dim), dtype=np.float32)


def _make_unique_dir(base_dir: Path) -> Path:
    if not base_dir.exists():
        base_dir.mkdir(parents=True, exist_ok=False)
        return base_dir
    idx = 1
    while True:
        candidate = Path(f"{str(base_dir)}{{{idx}}}")
        if not candidate.exists():
            candidate.mkdir(parents=True, exist_ok=False)
            return candidate
        idx += 1


def _obs_tree_max_abs_diff(a, b):
    if isinstance(a, dict):
        keys = sorted(set(a.keys()) & set(b.keys()))
        if len(keys) == 0:
            return 0.0
        return max(_obs_tree_max_abs_diff(a[k], b[k]) for k in keys)
    if isinstance(a, list):
        if len(a) == 0:
            return 0.0
        return max(_obs_tree_max_abs_diff(x, y) for x, y in zip(a, b))
    if isinstance(a, np.ndarray):
        if a.shape != b.shape:
            return float("inf")
        return float(np.max(np.abs(a - b)))
    try:
        return float(abs(float(a) - float(b)))
    except Exception:
        return 0.0 if a == b else float("inf")


def rollout_with_tracking(
    env,
    policy,
    init_state,
    guidance_fn,
    guidance_scale,
    seed,
    horizon,
    video_path,
    camera_name="frontview",
    video_skip=1,
    mujoco_body_name="Can_main",
):
    reseed(seed)
    policy.start_episode()
    obs = env.reset_to(init_state)

    observations = [_to_numpy_tree(obs)]
    states = [build_state_from_obs_dict(obs)]
    actions = []
    rewards = []
    dones = []
    success_flags = []

    rzz_mujoco = []
    can_pos_mujoco = []
    rzz0, pos0 = get_mujoco_rzz_and_pos(env, body_name=mujoco_body_name)
    rzz_mujoco.append(float(rzz0))
    can_pos_mujoco.append(np.asarray(pos0, dtype=np.float32))

    tracking = {
        "corrections": [],
        "diffusion": [],
        "guidance": [],
    }

    writer = imageio.get_writer(str(video_path), fps=20)
    total_reward = 0.0
    success = False

    try:
        for step_i in range(horizon):
            if float(guidance_scale) == 0.0:
                act = policy(
                    ob=obs,
                    guidance_function=None,
                    guidance_type=None,
                    guidance_scale=0.0,
                )
            else:
                act = policy(
                    ob=obs,
                    guidance_function=guidance_fn,
                    guidance_type="diffusion",
                    guidance_scale=float(guidance_scale),
                )

            inner = policy.policy
            tracking["corrections"].append(_clone_tracking_list(inner.corrections_list, inner))
            tracking["diffusion"].append(_clone_tracking_list(inner.diffusion_list, inner))
            tracking["guidance"].append(_clone_tracking_list(inner.guidance_list, inner))

            next_obs, r, done, _ = env.step(act)

            actions.append(np.asarray(act, dtype=np.float32))
            rewards.append(float(r))
            dones.append(bool(done))
            states.append(build_state_from_obs_dict(next_obs))
            observations.append(_to_numpy_tree(next_obs))

            rzz_now, pos_now = get_mujoco_rzz_and_pos(env, body_name=mujoco_body_name)
            rzz_mujoco.append(float(rzz_now))
            can_pos_mujoco.append(np.asarray(pos_now, dtype=np.float32))

            is_success = env.is_success()
            success = bool(is_success["task"]) if isinstance(is_success, dict) and "task" in is_success else bool(is_success)
            success_flags.append(success)

            total_reward += float(r)

            if step_i % video_skip == 0:
                frame = env.render(mode="rgb_array", height=512, width=512, camera_name=camera_name)
                writer.append_data(frame)

            if done or success:
                break

            obs = next_obs
    finally:
        writer.close()

    return {
        "guidance_scale": float(guidance_scale),
        "seed": int(seed),
        "horizon_requested": int(horizon),
        "steps_executed": int(len(actions)),
        "total_reward": float(total_reward),
        "success": bool(success),
        "actions": np.asarray(actions, dtype=np.float32),
        "rewards": np.asarray(rewards, dtype=np.float32),
        "dones": np.asarray(dones, dtype=np.bool_),
        "success_flags": np.asarray(success_flags, dtype=np.bool_),
        "states": np.asarray(states, dtype=np.float32),
        "observations": observations,
        "rzz_mujoco": np.asarray(rzz_mujoco, dtype=np.float32),
        "can_pos_mujoco": np.asarray(can_pos_mujoco, dtype=np.float32),
        "tracking": tracking,
        "video_path": str(video_path),
    }


# Ensure we have an init state
if "init_state" not in globals() or init_state is None:
    init_state_path = potential_seed_run_dirs[0] / "init_state.json"
    init_state = load_init_state_json(str(init_state_path))

# Reuse the world-model guidance function from the earlier sanity-check cell
guidance_fn = make_world_model_guidance_fn(
    predictor=predictor_learned,
    stats=stats,
    device=device,
    rollout_steps=8,
)

scales = [0, 10, 25, 50, 75, 100]
fixed_seed = 3 
horizon = 300
video_skip = 1
camera_name = "frontview"

output_root = _make_unique_dir(Path(f"../outputs/guidance_artifacts/seeded_guided_diffusion_seed_{fixed_seed}"))
print(f"Output directory: {output_root.resolve()}")

rollouts = {}
for scale in scales:
    scale_dir = output_root / f"scale_{scale}_seed_{fixed_seed}"
    scale_dir.mkdir(parents=True, exist_ok=False)

    video_path = scale_dir / f"rollout_scale_{scale}_seed_{fixed_seed}.mp4"
    print(f"Running rollout: scale={scale}, seed={fixed_seed}, horizon={horizon}")

    rollout_data = rollout_with_tracking(
        env=env,
        policy=policy,
        init_state=init_state,
        guidance_fn=guidance_fn,
        guidance_scale=scale,
        seed=fixed_seed,
        horizon=horizon,
        video_path=video_path,
        camera_name=camera_name,
        video_skip=video_skip,
    )

    rollouts[scale] = rollout_data

    with open(scale_dir / "rollout_full.pkl", "wb") as f:
        pickle.dump(rollout_data, f, protocol=pickle.HIGHEST_PROTOCOL)

    # Small numeric bundle for fast loading
    np.savez_compressed(
        scale_dir / "rollout_numeric.npz",
        actions=rollout_data["actions"],
        rewards=rollout_data["rewards"],
        dones=rollout_data["dones"],
        success_flags=rollout_data["success_flags"],
        states=rollout_data["states"],
        rzz_mujoco=rollout_data["rzz_mujoco"],
        can_pos_mujoco=rollout_data["can_pos_mujoco"],
    )

    metadata = {
        "scale": float(scale),
        "seed": fixed_seed,
        "horizon_requested": horizon,
        "steps_executed": int(rollout_data["steps_executed"]),
        "total_reward": float(rollout_data["total_reward"]),
        "success": bool(rollout_data["success"]),
        "video_path": str(video_path),
    }
    (scale_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))

In [ ]:

# ---------- Plot 1: per-env-timestep mean(|guidance|) for each scale ----------
plt.figure(figsize=(12, 5))
guidance_strength_series = {}

for scale in scales:
    per_timestep = []
    for g_step in rollouts[scale]["tracking"]["guidance"]:
        # g_step shape: [denoise_steps, 1, 16, 7]
        per_timestep.append(float(np.mean(np.abs(g_step))))
    guidance_strength_series[scale] = np.asarray(per_timestep, dtype=np.float32)

    plt.plot(
        np.arange(len(per_timestep)),
        per_timestep,
        linewidth=1.8,
        label=f"scale={scale}",
    )

plt.title("Mean |guidance term| per env timestep")
plt.xlabel("env timestep")
plt.ylabel("mean abs guidance")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plot1_path = output_root / "guidance_mean_abs_vs_timestep.png"
plt.savefig(plot1_path, dpi=120, bbox_inches="tight")
plt.show()


# ---------- Plot 2: Rzz for all scales on one plot ----------
plt.figure(figsize=(12, 5))
for scale in scales:
    rzz = rollouts[scale]["rzz_mujoco"]
    plt.plot(np.arange(len(rzz)), rzz, linewidth=2.0, label=f"scale={scale}")

plt.title("MuJoCo Rzz vs timestep (all scales)")
plt.xlabel("env timestep")
plt.ylabel("Rzz")
plt.ylim([0.91, 1.02])
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plot2_path = output_root / "rzz_all_scales.png"
plt.savefig(plot2_path, dpi=120, bbox_inches="tight")
plt.show()


# ---------- Plot 3: L2 norm of (action_guided - action_unguided) per timestep ----------
plt.figure(figsize=(12, 5))

ref_actions = rollouts[0]["actions"]  # scale 0 is unguided baseline
for scale in scales[1:]:  # Skip scale 0
    guided_actions = rollouts[scale]["actions"]
    
    # Compute L2 difference per timestep
    min_steps = min(len(ref_actions), len(guided_actions))
    l2_diffs = []
    for t in range(min_steps):
        diff = guided_actions[t] - ref_actions[t]
        l2 = float(np.linalg.norm(diff))
        l2_diffs.append(l2)
    
    plt.plot(np.arange(len(l2_diffs)), l2_diffs, linewidth=2.0, label=f"scale={scale} vs unguided")

plt.title("L2 norm of (action_guided - action_unguided) per timestep")
plt.xlabel("env timestep")
plt.ylabel("L2 action diff")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plot3_path = output_root / "action_divergence_from_unguided.png"
plt.savefig(plot3_path, dpi=120, bbox_inches="tight")
plt.show()


# ---------- Seed sanity checks ----------
print("\nSeed sanity checks")
print("-" * 80)

# 1) Initial observation equivalence across scales
ref_scale = scales[0]
ref_obs0 = rollouts[ref_scale]["observations"][0]
for scale in scales[1:]:
    obs0 = rollouts[scale]["observations"][0]
    max_abs_diff = _obs_tree_max_abs_diff(ref_obs0, obs0)
    print(f"Initial observation max abs diff (scale {ref_scale} vs {scale}): {max_abs_diff:.3e}")

# 2) Pre-guidance initial correction equivalence: env-step 0, denoise-step 0
ref_corr0 = rollouts[ref_scale]["tracking"]["corrections"][0][0]
for scale in scales[1:]:
    corr0 = rollouts[scale]["tracking"]["corrections"][0][0]
    l2 = float(np.linalg.norm(corr0 - ref_corr0))
    max_abs = float(np.max(np.abs(corr0 - ref_corr0)))
    print(f"Initial correction diff (scale {ref_scale} vs {scale}): L2={l2:.6e}, max_abs={max_abs:.6e}")

# 3) First action similarity check (should start similar; may diverge with stronger guidance)
ref_a0 = rollouts[ref_scale]["actions"][0]
for scale in scales[1:]:
    a0 = rollouts[scale]["actions"][0]
    l2 = float(np.linalg.norm(a0 - ref_a0))
    max_abs = float(np.max(np.abs(a0 - ref_a0)))
    print(f"First action diff (scale {ref_scale} vs {scale}): L2={l2:.6e}, max_abs={max_abs:.6e}")


# ---------- Save top-level experiment summary ----------
summary = {
    "output_root": str(output_root),
    "seed": fixed_seed,
    "scales": scales,
    "horizon": horizon,
    "plot_guidance_mean_abs": str(plot1_path),
    "plot_rzz": str(plot2_path),
    "per_scale": {
        str(s): {
            "steps_executed": int(rollouts[s]["steps_executed"]),
            "total_reward": float(rollouts[s]["total_reward"]),
            "success": bool(rollouts[s]["success"]),
            "video_path": str(rollouts[s]["video_path"]),
            "scale_dir": str(output_root / f"scale_{s}_seed_{fixed_seed}"),
        }
        for s in scales
    },
}
(output_root / "experiment_summary.json").write_text(json.dumps(summary, indent=2))

print("\nSaved artifacts:")
print(f"  Root: {output_root.resolve()}")
print(f"  Plot 1: {plot1_path}")
print(f"  Plot 2: {plot2_path}")
for s in scales:
    print(f"  Scale {s}: {(output_root / f'scale_{s}_seed_{fixed_seed}').resolve()}")